# Sentiment Analysis — Star-Rating Label Assignment



Labelling rule:
| Stars | Sentiment |
|-------|-----------|
| 1 – 2 | `negative` |
| 3     | `neutral`  |
| 4 – 5 | `positive` |

Reviews without a numeric rating (e.g. Reddit comments) are tagged `unrated` and kept for later text-based inference.

**Outputs**
- `sentiment_reviews.json` – flat list of all labelled reviews  
- `sentiment_by_product.json` – per-product sentiment summary + labelled reviews  
- `sentiment_summary.csv` – one row per product with counts & percentages

## 1. Imports & Paths

In [1]:
import re
import json
import pandas as pd
from pathlib import Path
from collections import Counter

# ── Paths ──────────────────────────────────────────────────────────────────────
# Adjust DRIVE_BASE if your folder layout differs
DRIVE_BASE   = Path("/content/drive/My Drive/CS_510/")
KB_PATH      = DRIVE_BASE / "knowledge_base.json"

OUT_REVIEWS  = DRIVE_BASE / "sentiment_reviews.json"       # flat review list
OUT_BY_PROD  = DRIVE_BASE / "sentiment_by_product.json"    # per-product detail
OUT_SUMMARY  = DRIVE_BASE / "sentiment_summary.csv"        # summary table

print(f"KB path  : {KB_PATH}")
print(f"KB exists: {KB_PATH.exists()}")

KB path  : /content/drive/My Drive/CS_510/knowledge_base.json
KB exists: False


## 2. Load Knowledge Base

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
with open(KB_PATH, encoding="utf-8") as f:
    knowledge_base = json.load(f)

total_products = len(knowledge_base)
total_reviews  = sum(e["total_reviews"] for e in knowledge_base.values())
print(f"Loaded {total_products} products | {total_reviews} total reviews")

Loaded 42 products | 4803 total reviews


## 3. Sentiment Labelling Function

In [5]:
def assign_sentiment(rating) -> str:
    """
    Map a numeric star rating to a sentiment label.

    Parameters
    ----------
    rating : float | int | None
        Star rating on a 1–5 scale, or None / NaN for unrated reviews.

    Returns
    -------
    str : 'positive' | 'neutral' | 'negative' | 'unrated'
    """
    if rating is None:
        return "unrated"
    try:
        r = float(rating)
    except (TypeError, ValueError):
        return "unrated"

    if r <= 2:
        return "negative"
    elif r == 3:
        return "neutral"
    else:                   # 4 or 5
        return "positive"


# Quick sanity-check
tests = [(1, "negative"), (2, "negative"), (3, "neutral"),
         (4, "positive"), (5, "positive"), (None, "unrated")]
for stars, expected in tests:
    result = assign_sentiment(stars)
    status = "✓" if result == expected else "✗"
    print(f"  {status}  stars={stars!s:<5} → {result}")

  ✓  stars=1     → negative
  ✓  stars=2     → negative
  ✓  stars=3     → neutral
  ✓  stars=4     → positive
  ✓  stars=5     → positive
  ✓  stars=None  → unrated


## 4. Apply Labels to All Reviews

In [6]:
all_labelled_reviews = []   # flat list used for downstream modelling
sentiment_by_product = {}   # per-product detail dict

for product_name, entry in knowledge_base.items():
    labelled = []

    for review in entry.get("all_reviews", []):
        sentiment = assign_sentiment(review.get("rating"))

        labelled_review = {
            "product_name": product_name,
            "source":       review.get("source", ""),
            "title":        review.get("title", ""),
            "body":         review.get("body", ""),
            "rating":       review.get("rating"),
            "sentiment":    sentiment,
        }
        labelled.append(labelled_review)

    # Sentiment counts for this product
    counts = Counter(r["sentiment"] for r in labelled)
    rated  = [r for r in labelled if r["sentiment"] != "unrated"]

    sentiment_by_product[product_name] = {
        "product_name":      product_name,
        "total_reviews":     len(labelled),
        "rated_reviews":     len(rated),
        "unrated_reviews":   counts.get("unrated", 0),
        "positive_count":    counts.get("positive", 0),
        "neutral_count":     counts.get("neutral",  0),
        "negative_count":    counts.get("negative", 0),
        "reviews":           labelled,
    }
    all_labelled_reviews.extend(labelled)

print(f"Labelled {len(all_labelled_reviews)} reviews across {len(sentiment_by_product)} products")

overall = Counter(r["sentiment"] for r in all_labelled_reviews)
print("\nOverall distribution:")
for label, cnt in sorted(overall.items()):
    pct = cnt / len(all_labelled_reviews) * 100 if all_labelled_reviews else 0
    print(f"  {label:<10}: {cnt:>5}  ({pct:.1f}%)")

Labelled 4803 reviews across 42 products

Overall distribution:
  negative  :  1427  (29.7%)
  neutral   :   304  (6.3%)
  positive  :  3062  (63.8%)
  unrated   :    10  (0.2%)


## 5. Build Summary DataFrame

In [7]:
summary_rows = []

for product_name, data in sentiment_by_product.items():
    rated = data["rated_reviews"]
    pos   = data["positive_count"]
    neu   = data["neutral_count"]
    neg   = data["negative_count"]

    summary_rows.append({
        "product_name":     product_name,
        "total_reviews":    data["total_reviews"],
        "rated_reviews":    rated,
        "unrated_reviews":  data["unrated_reviews"],
        "positive_count":   pos,
        "neutral_count":    neu,
        "negative_count":   neg,
        "pct_positive":     round(pos / rated * 100, 1) if rated else None,
        "pct_neutral":      round(neu / rated * 100, 1) if rated else None,
        "pct_negative":     round(neg / rated * 100, 1) if rated else None,
        # Dominant sentiment among rated reviews
        "dominant_sentiment": max(
            {"positive": pos, "neutral": neu, "negative": neg},
            key=lambda k: {"positive": pos, "neutral": neu, "negative": neg}[k]
        ) if rated else "n/a",
    })

summary_df = pd.DataFrame(summary_rows).sort_values("product_name")
print(f"Summary table: {summary_df.shape[0]} products × {summary_df.shape[1]} columns")
summary_df.head(10)

Summary table: 42 products × 11 columns


,product_name,total_reviews,rated_reviews,unrated_reviews,positive_count,neutral_count,negative_count,pct_positive,pct_neutral,pct_negative,dominant_sentiment
31,"Amara Organic Smoothie Melts, Mighty Sweet Gre...",0,0,0,0,0,0,NaN,NaN,NaN,n/a
0,Amazon Echo Dot (5th Gen),0,0,0,0,0,0,NaN,NaN,NaN,n/a
27,Anker 735 USB-C Charger (GaNPrime 65W),0,0,0,0,0,0,NaN,NaN,NaN,n/a
14,Apple Airpods 4,504,504,0,254,37,213,50.4,7.3,42.3,positive
7,Apple Watch Series 11,100,100,0,33,9,58,33.0,9.0,58.0,negative
5,Aquaphor Healing Ointment,0,0,0,0,0,0,NaN,NaN,NaN,n/a
29,Bedsure Satin Pillowcase Set,3,0,3,0,0,0,NaN,NaN,NaN,n/a
25,Bose QuietComfort 45 Headphones,0,0,0,0,0,0,NaN,NaN,NaN,n/a
35,Byoma Balancing Face Mist,5,5,0,3,1,1,60.0,20.0,20.0,positive
26,COSRX Snail Mucin 96% Power Repairing Essence,5,5,0,5,0,0,100.0,0.0,0.0,positive


## 6. Save Outputs

In [8]:
# 6a. Flat review list  (used as input for VADER / BERT in the next notebook)
with open(OUT_REVIEWS, "w", encoding="utf-8") as f:
    json.dump(all_labelled_reviews, f, indent=2, ensure_ascii=False)
print(f"Saved flat reviews  → {OUT_REVIEWS}")

# 6b. Per-product detail  (reviews stripped out for a lighter summary file)
summary_only = {
    k: {key: v for key, v in val.items() if key != "reviews"}
    for k, val in sentiment_by_product.items()
}
with open(OUT_BY_PROD, "w", encoding="utf-8") as f:
    json.dump(sentiment_by_product, f, indent=2, ensure_ascii=False)
print(f"Saved per-product   → {OUT_BY_PROD}")

# 6c. CSV summary
summary_df.to_csv(OUT_SUMMARY, index=False)
print(f"Saved CSV summary   → {OUT_SUMMARY}")

Saved flat reviews  → /content/drive/My Drive/CS_510/sentiment_reviews.json
Saved per-product   → /content/drive/My Drive/CS_510/sentiment_by_product.json
Saved CSV summary   → /content/drive/My Drive/CS_510/sentiment_summary.csv


## 7. Quick Inspection — Per-Product Breakdown

In [9]:
# Show the full summary table sorted by % negative (highest concern first)
summary_df.sort_values("pct_negative", ascending=False).reset_index(drop=True)

,product_name,total_reviews,rated_reviews,unrated_reviews,positive_count,neutral_count,negative_count,pct_positive,pct_neutral,pct_negative,dominant_sentiment
0,Keurig K-Mini Single Serve Coffee Maker,504,504,0,160,19,325,31.7,3.8,64.5,negative
1,Phillip Sonicare 4100 Series Electric Toothbrush,404,404,0,134,32,238,33.2,7.9,58.9,negative
2,Apple Watch Series 11,100,100,0,33,9,58,33.0,9.0,58.0,negative
3,Apple Airpods 4,504,504,0,254,37,213,50.4,7.3,42.3,positive
4,Dyson V8 Cordless Vacuum,404,404,0,197,51,156,48.8,12.6,38.6,positive
5,Owala Water Bottle,504,504,0,338,35,131,67.1,6.9,26.0,positive
6,Byoma Balancing Face Mist,5,5,0,3,1,1,60.0,20.0,20.0,positive
7,Milk Makeup Cooling Water Jelly Tint,5,5,0,2,2,1,40.0,40.0,20.0,positive
8,RX Bar Vanilla Almond Protein Bars,345,344,1,271,17,56,78.8,4.9,16.3,positive
9,Crocs,503,502,1,406,16,80,80.9,3.2,15.9,positive


In [10]:
# ── Drill into a specific product ─────────────────────────────────────────────
PRODUCT_TO_INSPECT = list(sentiment_by_product.keys())[0]   # change as needed

entry = sentiment_by_product[PRODUCT_TO_INSPECT]
print(f"Product : {entry['product_name']}")
print(f"  Total reviews  : {entry['total_reviews']}")
print(f"  Rated reviews  : {entry['rated_reviews']}")
print(f"  Positive       : {entry['positive_count']}")
print(f"  Neutral        : {entry['neutral_count']}")
print(f"  Negative       : {entry['negative_count']}")
print(f"  Unrated        : {entry['unrated_reviews']}")

print("\nSample reviews:")
for sentiment_label in ["positive", "neutral", "negative", "unrated"]:
    examples = [r for r in entry["reviews"] if r["sentiment"] == sentiment_label][:2]
    for r in examples:
        print(f"  [{r['sentiment'].upper():<8}] stars={r['rating']}  {r['body'][:120]}")

Product : Amazon Echo Dot (5th Gen)
  Total reviews  : 0
  Rated reviews  : 0
  Positive       : 0
  Neutral        : 0
  Negative       : 0
  Unrated        : 0

Sample reviews:


## 8. Coverage Check — How Many Reviews Lack a Rating?

Unrated reviews (mostly Reddit comments) will need text-based sentiment
inference in the next phase (VADER / fine-tuned BERT). This cell quantifies
the gap so you know how much labelling remains.

In [11]:
total    = len(all_labelled_reviews)
unrated  = sum(1 for r in all_labelled_reviews if r["sentiment"] == "unrated")
rated    = total - unrated

print(f"Total reviews   : {total}")
print(f"Rated (labelled): {rated}  ({rated/total*100:.1f}%)")
print(f"Unrated (Reddit/other): {unrated}  ({unrated/total*100:.1f}%)")
print()

# Products with the most unrated reviews
unrated_by_product = (
    summary_df[["product_name", "unrated_reviews", "total_reviews"]]
    .assign(pct_unrated=lambda d: (d["unrated_reviews"] / d["total_reviews"] * 100).round(1))
    .sort_values("unrated_reviews", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
print("Products with most unrated reviews:")
unrated_by_product

Total reviews   : 4803
Rated (labelled): 4793  (99.8%)
Unrated (Reddit/other): 10  (0.2%)

Products with most unrated reviews:


,product_name,unrated_reviews,total_reviews,pct_unrated
0,Mini Mic Pro,5,5,100.0
1,Bedsure Satin Pillowcase Set,3,3,100.0
2,Crocs,1,503,0.2
3,RX Bar Vanilla Almond Protein Bars,1,345,0.3
4,Amazon Echo Dot (5th Gen),0,0,NaN
5,Apple Airpods 4,0,504,0.0
6,Aquaphor Healing Ointment,0,0,NaN
7,Anker 735 USB-C Charger (GaNPrime 65W),0,0,NaN
8,Apple Watch Series 11,0,100,0.0
9,"Amara Organic Smoothie Melts, Mighty Sweet Gre...",0,0,NaN


## 9. Export `rating_UI.json`
Extracts only the sentiment percentage breakdown for each product — the exact shape the UI consumes.

In [12]:
OUT_RATING_UI = DRIVE_BASE / "rating_UI.json"

rating_ui = [
    {
        "id":          re.sub(r'[^a-z0-9]+', '-', product_name.lower()).strip('-'),
        "product_name": product_name,
        "sentiment": {
            "positive": round(data["positive_count"] / data["rated_reviews"] * 100) if data["rated_reviews"] else None,
            "neutral":  round(data["neutral_count"]  / data["rated_reviews"] * 100) if data["rated_reviews"] else None,
            "negative": round(data["negative_count"] / data["rated_reviews"] * 100) if data["rated_reviews"] else None,
        }
    }
    for product_name, data in sentiment_by_product.items()
]

with open(OUT_RATING_UI, "w", encoding="utf-8") as f:
    json.dump(rating_ui, f, indent=2, ensure_ascii=False)

print(f"Saved -> {OUT_RATING_UI}")
print(f"Entries: {len(rating_ui)}\n")
for p in rating_ui:
    print(f"  {p['product_name'][:45]:<45} {p['sentiment']}")

Saved -> /content/drive/My Drive/CS_510/rating_UI.json
Entries: 42

  Amazon Echo Dot (5th Gen)                     {'positive': None, 'neutral': None, 'negative': None}
  CeraVe Moisturizing Cream                     {'positive': None, 'neutral': None, 'negative': None}
  Neutrogena Hydro Boost Water Gel              {'positive': None, 'neutral': None, 'negative': None}
  Nespresso Vertuo Next Coffee Machine          {'positive': None, 'neutral': None, 'negative': None}
  OxiClean White Revive Laundry Whitener        {'positive': 80, 'neutral': 7, 'negative': 13}
  Aquaphor Healing Ointment                     {'positive': None, 'neutral': None, 'negative': None}
  Theragun Mini Percussive Massager             {'positive': None, 'neutral': None, 'negative': None}
  Apple Watch Series 11                         {'positive': 33, 'neutral': 9, 'negative': 58}
  Dyson V8 Cordless Vacuum                      {'positive': 49, 'neutral': 13, 'negative': 39}
  Fairlife Core Power High Protein